In [ ]:
from pathlib import Path
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from PIL import Image
from tqdm import tqdm
from yomitoku import OCR
from yomitoku.data.functions import load_image
from logging import getLogger

# ----- 日本語フォントを登録 -----
# IPAexGothic 等の .ttf を用意 (例: ipaexg.ttf)
FONT_PATH = "NotoSansJP-VariableFont_wght.ttf"
FONT_NAME = "IPAexGothic"
pdfmetrics.registerFont(TTFont(FONT_NAME, FONT_PATH))

# -----------------------------
# Mock OCR 関数（後で差し替え）
# -----------------------------
def mock_ocr(image_path: Path):
    """
    戻り値:
        List[Tuple[str, (x1, y1, x2, y2)]]
        座標は画像左上原点・px単位
    """
    return [
        ("This is a sample text", (100, 200, 600, 240)),
        ("OCR embedded text", (100, 260, 550, 300)),
    ]

# -----------------------------
# YomiToku OCR
# -----------------------------
def yomi_ocr_extract(image_path: Path, ocr_engine: OCR):
    """
    YomiToku OCR で文字位置＋認識結果を得る関数。
    戻り値:
        List of (text, (x1, y1, x2, y2)).
    """
    img = load_image(image_path)
    results, _ = ocr_engine(img[0])  # 公式 API: results は解析結果オブジェクト

    # 結果を text + bbox のリストに変換
    extracted = []
    
    for word in results.words:
        bbox = (word.points[0][0], word.points[0][1], word.points[2][0], word.points[2][1])
        extracted.append((word.content, bbox))

    return extracted

# -----------------------------
# JPGフォルダ → OCR付きPDF
# -----------------------------
def images_to_searchable_pdf(
    image_dir: Path,
    output_pdf: Path,
    device: str = "cuda",
):

    pdf = canvas.Canvas(str(output_pdf), pagesize=A4)
    page_w, page_h = A4

    # YomiToku OCR エンジン初期化
    ocr_engine = OCR(visualize=False, device=device)
    logger = getLogger('yomitoku.base')
    logger.disabled = True

    for img_path in tqdm(sorted(image_dir.glob("*.jpg")), desc="OCR & PDF", unit="page"):
        with Image.open(img_path) as img:
            img_w, img_h = img.size

        # ページ内の画像スケール
        scale = min(page_w / img_w, page_h / img_h)
        draw_w = img_w * scale
        draw_h = img_h * scale
        offset_x = (page_w - draw_w) / 2
        offset_y = (page_h - draw_h) / 2

        # 背景画像描画
        pdf.drawImage(
            str(img_path), offset_x, offset_y,
            width=draw_w, height=draw_h
        )

        # OCR JSON 経由で日本語テキスト取得
        ocr_lines = yomi_ocr_extract(img_path, ocr_engine)

        # 透明テキストとして埋め込み
        pdf.setFillAlpha(0)

        for text, (x1, y1, x2, y2) in ocr_lines:
            px = offset_x + x1 * scale
            py = offset_y + draw_h - y2 * scale

            # bbox からフォントサイズ（高さベース）
            font_size = max((y2 - y1) * scale * 0.8, 4)
            pdf.setFont(FONT_NAME, font_size)

            pdf.drawString(px, py, text)

        pdf.setFillAlpha(1)
        pdf.showPage()

    pdf.save()

# -----------------------------
# 実行例
# -----------------------------
if __name__ == "__main__":
    
    image_dir_path = Path("./input_data/深層学習教科書 ディープラーニング E資格（エンジニア）精選問題集/")
    output_pdf_path = Path("./output_data/") / f"{image_dir_path.stem}.pdf"

    images_to_searchable_pdf(
        image_dir=image_dir_path,
        output_pdf=output_pdf_path,
    )


2025-12-14 16:25:21,949 - yomitoku.base - INFO - Initialize TextDetector
2025-12-14 16:25:22,842 - yomitoku.base - INFO - Initialize TextRecognizer
OCR & PDF:   0%|          | 0/546 [00:00<?, ?page/s]2025-12-14 16:25:24,299 - yomitoku.base - INFO - TextDetector __call__ elapsed_time: 0.44994568824768066
2025-12-14 16:25:24,552 - yomitoku.base - INFO - TextRecognizer __call__ elapsed_time: 0.2527909278869629
OCR & PDF:   0%|          | 0/546 [00:00<?, ?page/s]
